# Classes et héritage

## Héritage simple

Créez une classe `ModelUn`  retournant une constante fixée lors de l'initialisation :

In [2]:
import numpy as np

class ModelUn():
    def __init__(self, init_val: int = 42) -> None:
        self.prediction_value = init_val
        print("Model 1 initialisation")
        
    def predict(self, X: np.ndarray) -> int:
        print("Model 1  prediction")
        return self.prediction_value

In [3]:
clf = ModelUn(43)
print(f'prediction : {clf.predict(1)}')

Model 1 initialisation
Model 1  prediction
prediction : 43


**Remarque :** la notation ci-dessous est équivalente

In [4]:
print(f'prediction : {ModelUn.predict(clf, 1)}')

Model 1  prediction
prediction : 43


On souhaite étendre le comportement de la classe `ModelUn` pour un nouveau projet, sans modifier le code existant. Créez une seconde classe `ModelDeux` héritant de `ModelUn` et stockant un entier supplémentaire lors de l'initilisation. La méthode `predict` de `ModelDeux` doit retourner la liste des deux entiers stockés. Pour cela, elle doit appeler les méthodes de la classe parent (`ModelUn`), et compléter leur comportement.

In [5]:
class ModelDeux(ModelUn):
    def __init__(self, init_val: int = 42, init_val2: int = 43) -> None:
        self.prediction_value2 = init_val2
        print("Model 2 initialisation")
        super().__init__(init_val)
        
    def predict(self, X: np.ndarray) -> int:
        print("Model 2 prediction")
        first_value = super().predict(X)
        return [first_value, self.prediction_value2]

In [6]:
clf = ModelDeux(1,2)
print("--------")
clf.predict(1)

Model 2 initialisation
Model 1 initialisation
--------
Model 2 prediction
Model 1  prediction


[1, 2]

Remarque :
- `super` n'appelle pas nécessairement la classe parent. En effet, Python utilise le MRO (_Method Resolution Order_) pour savoir quelle méthode appeler.
- `super().predict(X)` et `ModelUn.predict(self, X)` ont le même effet dans l'exemple précédent car il s'agit d'héritage simple.
- Les deux appels ne sont pas complètement équivalents :
    - `super().predict(X)` laisse Python trouver quelle est la méthode à appeler (via le MRO).
    - `ModelUn.predict(self, X)` l'impose (la méthode `predict` de `ModelUn`).

## Héritage multiple

On souhaite rajouter un entraînement sans modifier la classe existante. Le modèle stocke la valeur lue lors de l'entraînement et la rajoute à la liste de la méthode `predict`. 

In [7]:
class ModelTrois(ModelUn):       
    def fit(self, X: int) -> None:
        print("Model 3 fit")
        self.learnt_value = X
            
    def predict(self, X: np.ndarray) -> int:
        print("Model 3 prediction")
        first_value = super().predict(X)
        return [first_value, self.learnt_value]

In [8]:
clf = ModelTrois(3)
print("--------")
clf.fit(2)
print("--------")
clf.predict(1)

Model 1 initialisation
--------
Model 3 fit
--------
Model 3 prediction
Model 1  prediction


[3, 2]

Explication :
- Python commence par chercher le constructeur (`__init__`) de la classe `ModelTrois`
- Comme celui-ci n'existe pas, il remonte à sa classe parent (`ModelUn`)
- C'est donc `ModelUn.__init__` qui est appelé !
- La méthode `predict` de `ModelTrois` appelle celle de `ModelUn`

In [9]:
class ModelQuatre(ModelDeux, ModelTrois):
    None

In [10]:
clf = ModelQuatre(33, 34)
print("--------")
clf.fit(2)
print("--------")
clf.predict(3)

Model 2 initialisation
Model 1 initialisation
--------
Model 3 fit
--------
Model 2 prediction
Model 3 prediction
Model 1  prediction


[[33, 2], 34]

On peut afficher les attributs de l'objet `clf` :

In [11]:
clf.__dict__

{'prediction_value2': 34, 'prediction_value': 33, 'learnt_value': 2}

Et même le MRO de `ModelQuatre` :

In [12]:
ModelQuatre.__mro__

(__main__.ModelQuatre,
 __main__.ModelDeux,
 __main__.ModelTrois,
 __main__.ModelUn,
 object)

Explications :
- Concernant l'instanciation de `clf` :
    - `ModelQuatre` n'ayant pas de `__init__`, Python regarde dans l'ordre de déclarations des parents
    - C'est d'abord `ModelDeux` qui possède un `__init__`, Python l'exécute
        - `clf.prediction_value2` vaut donc 34
        - la ligne `super().__init__` n'appelle pas le `__init__` de `ModelUn` mais celui de `ModelTrois` (suivant dans le MRO de `ModelQuatre`
        - `ModelTrois` n'ayant pas `__init__`, c'est finalement celui de `ModelUn` (le suivant dans la liste du MRO de `ModelQuatre`!) qui est appelé. On a donc `clf.prediction_value` qui vaut 33

- Concernant le `clf.fit(2)`, `ModelTrois` est la première classe du MRO à posséder une méthode `fit`. C'est donc celle-ci qui est appelée, créant un attribut `learnt_value` valant 2 à `clf.`

- Concernant le `clf.predict(3)`, on appelle la méthode `predict` de `ModelDeux` (première classe dans l'ordre du MRO à la posséder). Le `super().predict(X)` de `ModelDeux` appelle la méthode `predict` de `ModelTrois` (classe suivant dans le MRO). Puis, à son tour, `super().predict(X)` de `ModelTrois` appelle la méthode `predict` de `ModelUn` qui renvoie `self.prediction_value` (33). La méthode `predict` de `ModelTrois` renvoie ensuite `[33, self.learnt_value]` (soit [33, 2]). Enfin, `predict` de `ModelDeux` renvoie `[[33, 2], self.prediction_value2]` soit [[33, 2], 34].

Dans la classe `ModelQuatre`, `super()` est en réalité `super(ModelQuatre, self)` :
- Le premier paramètre indique où on est dans le MRO, le deuxieme l'objet auquel prendre le MRO.
- Si le deuxième paramètre est une instance, on peut appeler `predict(X)`.
- si le deuxième paramètre est une classe, il faut appeler `predict(self, X)`.

Exemples :
- `super(ModelDeux, clf).predict(X)` va choisir la méthode `predict` de `ModelTrois` car `ModelTrois` suit `ModelDeux` dans le MRO de `clf` qui est une instance de `ModelQuatre`.
- `super(ModelDeux, ModelDeux).predict(clf, X)` va appeler `ModelUn`, car on se place dans `ModelDeux` mais on appelle le MRO de `ModelDeux`.

`super(ModelDeux, clf).predict(X)` appelle `predict` de `ModelTrois` et non la méthode de `ModelUn` comme on pourrait le penser (car `ModelDeux` hérite de `ModelUn`). 

Comment modifier `ModelDeux` et `ModelQuatre` pour que chaque modèle parent (`ModelDeux` et `ModelTrois`) appelle la méthode `predict` de `ModelUn` de manière isolée et indépendante, sans que l'appel de l'un n'entraîne automatiquement l'appel de l'autre via le MRO global ?

**Solution :** Changer le code de `ModelDeux` pour ne pas laisser à `super()` le choix (nen suivant le MRO de `clf`, c'est-à-dire de `ModelQuatre`) pour résoudre le nom :

In [15]:
class ModelDeux(ModelUn):
    def __init__(self, init_val: int = 42, init_val2: int=43) -> None:
        self.prediction_value2 = init_val2
        print("Model 2 initialisation")
        super().__init__(init_val)
        
    def predict(self, X:np.ndarray) -> int:
        print("Model 2 prediction")
        # ligne equivalente : first_value = ModelUn.predict(self, X)
        first_value = super().predict(X)
        return [first_value, self.prediction_value2]

In [16]:
class ModelQuatre(ModelDeux, ModelTrois):
    def predict(self, X):
        # appelle ModelTrois qui est le suivant selon le MRO de self lorsque l'on est dans ModelDeux
        res_model3 = super().predict(X)
        print("Appel de model 3 terminé")
        # équivalent à super(ModelQuatre, self).predict(X) donc on appelle predict de ModelDeux
        res_model2 = super(ModelQuatre, self).predict(X)
        print("Appel de model 2 terminé")
        return res_model3 + res_model2

In [17]:
clf = ModelQuatre(33, 34)
print("--------")
clf.fit(2)
print("--------")
clf.predict(3)

Model 2 initialisation
Model 1 initialisation
--------
Model 3 fit
--------
Model 2 prediction
Model 3 prediction
Model 1  prediction
Appel de model 3 terminé
Model 2 prediction
Model 3 prediction
Model 1  prediction
Appel de model 2 terminé


[[33, 2], 34, [33, 2], 34]

### La morale :

- Ne jamais utiliser `super()` en dehors du constructeur sauf si vous savez vraiment pourquoi c'est nécessaire. Préferez `ClassParent.method(self, params)`
- Toujours appeler `super()` dans le constructeur, de façon à ce que le constructeur de chaque classe ne soit appelé qu'une fois
- Si vous voulez plus d'infos, vous pouvez consulter ce très bon article (en anglais) : [https://fuhm.net/super-harmful/](https://fuhm.net/super-harmful/)

# Héritage dynamique

Imaginons que nous souhaitions créer une classe `ModelCinq` pouvant hériter soit de `ModelUn`, soit de `ModelDeux` et posssédant une méthode `fit`. Comment faire ?

Il s'agit d'un problème courant quand on crée des expériences en fonction de fichiers de configuration. La solution consiste à implémenter une méthode qui construit la classe en fonction de la classe parent demandée (fournie en paramètre) :

In [13]:
def model_cinq_builder(parent_class):
    class ModelCinq(parent_class):
        def fit(self, X: int):
            print("ModelCinq fit")
            return (X)
    return ModelCinq

In [14]:
print("---------- Instance with model 1 parent ------")
clf = model_cinq_builder(ModelUn)(33)
print("--------")
print(clf.fit(2))
print("--------")
print(clf.predict(3))

print("\n-------- Instance with model 2 parent --------")
clf = model_cinq_builder(ModelDeux)(33, 34)
print("--------")
print(clf.fit(2))
print("--------")
print(clf.predict(3))

---------- Instance with model 1 parent ------
Model 1 initialisation
--------
ModelCinq fit
2
--------
Model 1  prediction
33

-------- Instance with model 2 parent --------
Model 2 initialisation
Model 1 initialisation
--------
ModelCinq fit
2
--------
Model 2 prediction
Model 1  prediction
[33, 34]


# Paramètres dynamiques

Comment faire une méthode qui peut accepter un nombre variable de paramètres ?

Il s'agit d'un problème récurrent lorsqu'on fait des expériences décrites par des fichiers ! 

In [24]:
def my_method(**args):
    for key, value in args.items():
        print(f"Variables {key} de valeur : {value}")

In [25]:
my_method(deux=2, trois=3, quaranteDeux='toto')

Variables deux de valeur : 2
Variables trois de valeur : 3
Variables quaranteDeux de valeur : toto


Comment faire pour stocker dynamiquement un nombre de paramètres variable dans l'`__init__` d'une classe ?

In [28]:
class DynClass():
    def __init__(self, **args):
        for key, value in args.items():
            self.__dict__[key] = value
        self.static_param = 4


In [29]:
my_cls = DynClass(deux=2, trois=3, quaranteDeux='toto')

if hasattr(my_cls, "deux"):
    print(my_cls.deux)

if hasattr(my_cls, "trois"):
    print(my_cls.trois)

if not hasattr(my_cls, "un"):
    print(f"my_cls n'a pas d'attribut 'un'")

2
3
my_cls n'a pas d'attribut 'un'


Il s'agit d'une bonne pratique pour ne pas avoir à modifier un code si on sait que le nom ou la quantité de paramètres change tout le temps. De plus, c'est souvent nécessaire pour la méthode `__init__` pour qu'elle soit compatible avec `super()`.

Les classes en Python fonctionnent en fait avec un dictionnaire qui contient tous leurs paramètres :

In [30]:
my_cls.__dict__

{'deux': 2, 'trois': 3, 'quaranteDeux': 'toto', 'static_param': 4}

# Bonus : affichage d'une classe

Voici ce qui s'affiche par défaut :

In [31]:
my_cls = DynClass(deux=2, trois=3, quaranteDeux='toto')
print(my_cls)

In [32]:
my_cls

Cela ne nous apporte pas beaucoup d'information ! Le méthodes `__repr__` et `__str__` peuvent nous permettre d'afficher des informations pertinentes à la place :

In [33]:
class DynClass():
    def __init__(self, **args):
        for key, value in args.items():
            self.__dict__[key] = value
        self.static_param = 4
    
    def __repr__(self):
        params = [f'Parameter {k}: {v}' for k, v in self.__dict__.items()]
        return "\n".join(params)
    
    def __str__(self):
        return f'What a nice {self.__class__.__name__} with {len(self.__dict__.keys())} parameters!'

In [34]:
my_cls = DynClass(deux=2, trois=3, quaranteDeux='toto')
print(my_cls)

What a nice DynClass with 4 parameters!


In [35]:
my_cls

Parameter deux: 2
Parameter trois: 3
Parameter quaranteDeux: toto
Parameter static_param: 4